# 第17章：接入已有 Python 程序

本篇依据[本章课件](https://github.com/xbsheng/atguigu-note/blob/bbe79814defe76ae597d07dd4f53f2e1fb26ed5d/agent_skills/%E8%AF%BE%E4%BB%B6/%E5%AE%9E%E6%88%98%E7%AF%87/03-%E6%8E%A5%E5%85%A5%E5%B7%B2%E6%9C%89Python%E7%A8%8B%E5%BA%8F.md)整理，对应[视频 P28](https://www.bilibili.com/video/BV14rxM6qEnL?p=28)。资料快照：`bbe79814defe`（2026-10-29）。

以下命令、提示词和示例为课程阅读内容；样本数字是预期核对值，未在本次笔记整理中运行课程服务、安装技能或执行审批。

[完整配套工程](https://github.com/xbsheng/atguigu-note/tree/bbe79814defe76ae597d07dd4f53f2e1fb26ed5d/agent_skills/%E4%BB%A3%E7%A0%81/ecommerce-agent)；[业务口径](https://github.com/xbsheng/atguigu-note/blob/bbe79814defe76ae597d07dd4f53f2e1fb26ed5d/agent_skills/%E8%AF%BE%E4%BB%B6/%E5%AE%9E%E6%88%98%E7%AF%87/%E8%B5%84%E6%96%99/%E4%B8%9A%E5%8A%A1%E9%9C%80%E6%B1%82%E4%B8%8E%E6%95%B0%E6%8D%AE%E5%8F%A3%E5%BE%84.md)；[验收场景](https://github.com/xbsheng/atguigu-note/blob/bbe79814defe76ae597d07dd4f53f2e1fb26ed5d/agent_skills/%E8%AF%BE%E4%BB%B6/%E5%AE%9E%E6%88%98%E7%AF%87/%E8%B5%84%E6%96%99/%E9%AA%8C%E6%94%B6%E5%9C%BA%E6%99%AF%E4%B8%8E%E9%A2%84%E6%9C%9F%E7%BB%93%E6%9E%9C.md)。

## 17.1 先确认原程序职责

app/checker.py检查离线订单结构，不查数据库、不判断超时。CLI与MCP复用check_orders，接入前明确输入、输出、数据问题与执行失败的区别。

## 17.2 数据契约

最多1000条对象的JSON数组，id为非空字符串且批次唯一，status限定paid/shipped/cancelled，paid_at为带时区ISO时间，quantity为正整数。数字2和字符串"2"不同；布尔值也不能当整数数量。

```json
[{"id":"DEMO001","status":"paid","paid_at":"2026-10-10T08:00:00+08:00","quantity":2}]
```

时间偏移避免跨地区解释歧义，错误由校验器报告，模型不静默改输入。

### 校验函数怎样累计问题

每条记录按数组顺序给出row，从1开始；检查id、status、quantity、paid_at，并将同一条的多个错误放进errors。error_count汇总错误条目数，record_count仍是输入记录总数，二者不是同一单位。

错误输入不会被模型修成看似有效的值。空数组满足该校验器“最多1000条”的约定；而补货和巡检程序要求非空，是另一业务契约。不同工具不能因都接JSON就共享所有边界规则。

## 17.3 CLI先独立验证

在ecommerce-agent根目录：

```powershell
uv run --frozen python -m app.checker samples/orders.json
$LASTEXITCODE
```

正常样本3条。结果字段为valid、record_count、error_count、issues；问题序号从1开始，error_count是各问题字段数总和，可能多于有错记录数。

退出0为有效，1为数据有问题，2为读取/解析执行失败。业务结果JSON写stdout，执行错误写stderr。空数组符合该校验器的最多1000条契约；第十二章补货程序要求非空，不能通用推断。

## 17.4 包装成MCP工具

`ecommerce_check_orders(records)`直接接收数组并调用check_orders。工具层提供名称、参数描述与结构化返回，不让模型拼任意命令或传任意本地路径。CLI负责读取文件，业务函数只处理数据。

工具定义在mcp_server.py，返回注解 `dict[str, Any]` 对本项目锁定SDK的结构化输出有意义。READ/WRITE提示帮助客户端认识副作用，真正业务授权仍由API检查。

### 为什么工具直接接records

CLI需要从文件读取，工具调用已经能提交结构化数组，两种入口复用同一check_orders即可。MCP工具不接任意路径或拼接命令，减少模型必须理解的本地运行细节，返回值也保持可程序化读取。

FastMCP通过函数签名、描述和Pydantic Field约束生成工具Schema，records最多1000条先在工具层约束，业务函数仍校验自己的输入。READ等注解描述副作用，不授予访问业务数据的权限。

## 17.5 Streamable HTTP接入Codex

| 入口 | 用途 |
| --- | --- |
| 127.0.0.1:8000/ | 前端页面 |
| 127.0.0.1:8000/orders | 普通REST业务接口 |
| 127.0.0.1:8001/mcp | MCP协议端点 |

第一步在工程目录新开终端启动MCP，API可继续在另一个终端运行：

```powershell
$env:AGENT_TOKEN = '替换成API端相同的Agent令牌'
$env:API_BASE_URL = 'http://127.0.0.1:8000'
uv run --frozen python -m app.mcp_http
```

MCP只读Agent令牌与API地址，不加载.env里的数据库和审批凭据。单独本地校验工具不依赖API，其他查询/工作流工具依赖8000服务。改端口时可追加 `--port 8002` 并同步连接URL。

第二步合并用户Codex配置；已有同名节替换，旧stdio的command/args/cwd/env_vars移除，其他设置保留：

```toml
[mcp_servers.ecommerce]
url = "http://127.0.0.1:8001/mcp"
bearer_token_env_var = "AGENT_TOKEN"
startup_timeout_sec = 20
tool_timeout_sec = 30
```

第三步从具有相同连接令牌的终端启动Codex：

```powershell
$env:AGENT_TOKEN = '替换成MCP端相同的Agent令牌'
codex mcp list
codex
```

HTTP下客户端不启动Python子进程，不需要API_BASE_URL。CLI/IDE/桌面进程环境不同，变更后让相应进程获取变量并重启。此例为本地Bearer配置，无需用OAuth的codex mcp login。

配置列表只显示登记，进入CLI用/mcp检查连接与六工具。令牌缺失/不符为401，未启动是连接错误。直接浏览/mcp不能按普通网页判断协议是否可用。

第四步让Agent实际调用 `ecommerce_check_orders`，传入本篇DEMO001数组，核对1条、valid=true、error_count=0及执行轨迹。

### 三个进程的配置分别属于谁

API进程读取业务库和审批配置；MCP进程只拿AGENT_TOKEN与API_BASE_URL；Codex进程只需MCP URL及连接令牌。HTTP客户端不会替服务端启动Python，也不能通过自己设置API_BASE_URL改变已经运行的MCP服务环境。

mcp_http创建无状态HTTP应用并检查Bearer令牌，复用mcp_server中的同一工具对象。无状态是协议连接层属性，工作流的业务状态仍在API及其数据库/检查点里。

## 17.6 异常返回

把quantity改0，应返回未通过与数量问题，不自动修复；两条同id触发重复。工具不在线或参数不满足工具Schema是调用失败，不能记为校验通过。

## 17.7 复用函数的稳定契约

原程序负责确定性判断，MCP暴露能力，Codex解释结果。网络连接状态不保存业务工作流，后续仍用workflow_id查询持久化状态，MCP重连不重新建流程。

## 17.8 三个源文件的阅读顺序

以下依次保留checker.py、共用工具定义mcp_server.py与HTTP入口mcp_http.py。HTTP入口复用同一个mcp对象，设stateless_http和json_response，添加Bearer校验中间件，最后由Uvicorn运行；stdio入口仍用于配套测试。

工具适配的request统一处理API调用与错误，写操作超时属于结果不确定，后续要按同request_id查询或重试，不能声称未执行。

### 阅读request函数的失败分支

request统一构造Authorization、10秒超时和业务URL；正常返回解析JSON，4xx尽量保留本服务detail，5xx不向模型暴露内部堆栈或代理页。连接错误说明没有取得业务响应，超时则特别提醒写入结果可能不确定。

读操作可在条件清楚时重试；启动工作流写入超时，要保持request_id一致查询或重试，避免以新编号再次启动。错误不转换为空列表，模型才不会把查询失败描述成零异常。

## 17.9 checker.py完整代码

CLI和MCP共同调用该函数，避免重复实现。

[配套文件原文](https://github.com/xbsheng/atguigu-note/blob/bbe79814defe76ae597d07dd4f53f2e1fb26ed5d/agent_skills/%E4%BB%A3%E7%A0%81/ecommerce-agent/app/checker.py)。代码按资料快照保留。

```python
"""已有 Python 程序：校验订单基础字段，不依赖 MCP、数据库或模型。"""
import json
import sys
from datetime import datetime
from pathlib import Path


def check_orders(records: list) -> dict:
    if not isinstance(records, list) or len(records) > 1000:
        raise ValueError("输入必须为最多 1000 条记录的数组")
    issues = []
    seen = set()
    for row, record in enumerate(records, 1):
        errors = []
        if not isinstance(record, dict):
            issues.append({"row": row, "errors": ["记录必须为对象"]})
            continue
        order_id = record.get("id")
        if not isinstance(order_id, str) or not order_id.strip():
            errors.append("id 必须为非空字符串")
        elif order_id in seen:
            errors.append("id 重复")
        else:
            seen.add(order_id)
        if record.get("status") not in ("paid", "shipped", "cancelled"):
            errors.append("status 必须为 paid/shipped/cancelled")
        quantity = record.get("quantity")
        if type(quantity) is not int or quantity <= 0:
            errors.append("quantity 必须为正整数")
        try:
            paid_at = datetime.fromisoformat(record["paid_at"])
            if paid_at.utcoffset() is None:
                raise ValueError("缺少时区")
        except (KeyError, TypeError, ValueError):
            errors.append("paid_at 必须为带时区的 ISO 时间")
        if errors:
            issues.append({"row": row, "errors": errors})
    return {"valid": not issues, "record_count": len(records),
            "error_count": sum(len(i["errors"]) for i in issues), "issues": issues}


def main() -> int:
    try:
        if len(sys.argv) != 2:
            raise ValueError("用法：python -m app.checker INPUT.json")
        records = json.loads(Path(sys.argv[1]).read_text(encoding="utf-8-sig"))
        result = check_orders(records)
        print(json.dumps(result, ensure_ascii=False, indent=2))
        return 0 if result["valid"] else 1
    except (OSError, ValueError) as error:
        print(json.dumps({"execution_error": str(error)}, ensure_ascii=False), file=sys.stderr)
        return 2


if __name__ == "__main__":
    sys.exit(main())
```

## 17.10 mcp_server.py完整代码

包含六项工具、参数范围、HTTP适配及副作用注解。

[配套文件原文](https://github.com/xbsheng/atguigu-note/blob/bbe79814defe76ae597d07dd4f53f2e1fb26ed5d/agent_skills/%E4%BB%A3%E7%A0%81/ecommerce-agent/app/mcp_server.py)。代码按资料快照保留。

```python
"""stdio 与 HTTP 共用的 MCP 工具定义。只读取 Agent 凭据，不加载 .env。"""
import os
from typing import Annotated, Any
from urllib.parse import quote
import httpx
from mcp.server.fastmcp import FastMCP
from pydantic import Field
from app.checker import check_orders

mcp = FastMCP("ecommerce_mcp")
READ = {"readOnlyHint": True, "destructiveHint": False, "idempotentHint": True, "openWorldHint": True}
WRITE = {"readOnlyHint": False, "destructiveHint": False, "idempotentHint": True, "openWorldHint": True}
ShortText = Annotated[str, Field(min_length=1, max_length=80)]


async def request(method: str, path: str, **kwargs) -> dict[str, Any]:
    token = os.getenv("AGENT_TOKEN")
    if not token:
        raise RuntimeError("缺少 AGENT_TOKEN，请在 MCP 服务进程环境中配置 Agent 令牌")
    url = os.getenv("API_BASE_URL", "http://127.0.0.1:8000").rstrip("/")
    try:
        async with httpx.AsyncClient(timeout=10, trust_env=False) as client:
            response = await client.request(method, url + path,
                headers={"Authorization": f"Bearer {token}"}, **kwargs)
        if response.is_error:
            # 不回传代理页、内部堆栈或凭据；只转发本服务的业务 detail。
            detail = "服务调用失败，请检查 API 日志"
            if response.status_code < 500:
                try:
                    detail = response.json().get("detail", detail)
                except ValueError:
                    pass
            raise RuntimeError(f"业务 API HTTP {response.status_code}: {detail}")
        return response.json()
    except httpx.TimeoutException as error:
        raise RuntimeError("业务 API 超时；写操作请使用同一 request_id 查询或重试，不能断言未执行") from error
    except httpx.RequestError as error:
        raise RuntimeError("无法连接业务 API，请检查 API_BASE_URL 与服务状态") from error


@mcp.tool(annotations={**READ, "openWorldHint": False})
def ecommerce_check_orders(records: Annotated[list[dict], Field(max_length=1000)]) -> dict[str, Any]:
    """校验最多1000条订单基础字段；返回 valid、record_count、error_count、issues。数据错误不等于执行失败。"""
    return check_orders(records)


@mcp.tool(annotations=READ)
async def ecommerce_list_orders(paid_on: ShortText, as_of: ShortText,
    offset: Annotated[int, Field(ge=0)] = 0,
    limit: Annotated[int, Field(ge=1, le=100)] = 20) -> dict[str, Any]:
    """分页查询授权店铺已付款且超过24小时未发货的订单。paid_on按上海日期，as_of须带时区；返回total/items/next_offset，须读取全部页。"""
    return await request("GET", "/orders", params=dict(paid_on=paid_on, as_of=as_of, offset=offset, limit=limit))


@mcp.tool(annotations=READ)
async def ecommerce_order_evidence(order_id: ShortText) -> dict[str, Any]:
    """获取授权订单的order、inventory、fulfillment、analysis。库存为当前快照，possible_reason不等于已证实历史根因。"""
    return await request("GET", f"/orders/{quote(order_id, safe='')}/evidence")


@mcp.tool(annotations=WRITE)
async def ecommerce_start_workflow(paid_on: ShortText, as_of: ShortText, request_id: ShortText) -> dict[str, Any]:
    """启动订单异常排查并暂停待审批，返回workflow_id/status/report/tickets；request_id重试须保持不变。最多100单。不会自动批准。"""
    return await request("POST", "/workflows", json=dict(paid_on=paid_on, as_of=as_of, request_id=request_id))


@mcp.tool(annotations=READ)
async def ecommerce_workflow_status(workflow_id: ShortText) -> dict[str, Any]:
    """查询持久化工作流状态、报告、审批决策与工单结果，不推进执行。"""
    return await request("GET", f"/workflows/{quote(workflow_id, safe='')}")


@mcp.tool(annotations=WRITE)
async def ecommerce_resume_workflow(workflow_id: ShortText) -> dict[str, Any]:
    """仅按服务端已有人工审批记录恢复工作流；无审批返回409。批准后创建工单，拒绝后结束；重复恢复不重复建单。"""
    return await request("POST", f"/workflows/{quote(workflow_id, safe='')}/resume")


if __name__ == "__main__":
    mcp.run(transport="stdio")
```

## 17.11 mcp_http.py完整代码

独立HTTP进程，不读取服务端.env；本地同一Agent令牌的教学配置不等于企业多用户认证。

[配套文件原文](https://github.com/xbsheng/atguigu-note/blob/bbe79814defe76ae597d07dd4f53f2e1fb26ed5d/agent_skills/%E4%BB%A3%E7%A0%81/ecommerce-agent/app/mcp_http.py)。代码按资料快照保留。

```python
"""本地 Streamable HTTP MCP 入口；复用现有六项工具，不加载 .env。"""
import argparse
import os
import secrets
import uvicorn
from starlette.responses import JSONResponse
from app.mcp_server import mcp


class AgentTokenMiddleware:
    """仅接受本地教学 Agent 令牌；不提供 OAuth 登录。"""

    def __init__(self, app, token: str):
        self.app = app
        self.token = token.encode("utf-8")

    async def __call__(self, scope, receive, send):
        if scope["type"] == "http":
            authorization = dict(scope["headers"]).get(b"authorization", b"")
            scheme, _, token = authorization.partition(b" ")
            if scheme.lower() != b"bearer" or not secrets.compare_digest(token, self.token):
                response = JSONResponse(
                    {"detail": "缺少有效 MCP Agent 令牌"},
                    status_code=401,
                    headers={"WWW-Authenticate": "Bearer"},
                )
                await response(scope, receive, send)
                return
        await self.app(scope, receive, send)


def create_http_app():
    token = os.getenv("AGENT_TOKEN", "")
    if not token or token.startswith("replace-"):
        raise RuntimeError("请在启动 MCP HTTP 服务的终端设置非占位 AGENT_TOKEN")
    mcp.settings.stateless_http = True
    mcp.settings.json_response = True
    app = mcp.streamable_http_app()
    app.add_middleware(AgentTokenMiddleware, token=token)
    return app


def main():
    parser = argparse.ArgumentParser(description="启动本机 MCP Streamable HTTP 服务")
    parser.add_argument("--port", type=int, default=8001)
    args = parser.parse_args()
    if not 1 <= args.port <= 65535:
        parser.error("端口必须在 1—65535 范围内")
    uvicorn.run(create_http_app(), host="127.0.0.1", port=args.port, workers=1)


if __name__ == "__main__":
    main()
```